# ICS40125 - Laboratorio N°09

Clasificación de tumores mamarios: pipeline completo de ML supervisado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc)

%matplotlib inline
sns.set_palette("deep", desat=0.6)

df = pd.read_csv("https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/BC.csv")
df.set_index('id', inplace=True)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0}).astype(int)
df.head()

## 1. Análisis exploratorio (EDA)

In [ ]:
print("Dimensiones:", df.shape)
print("Nulos:", df.isnull().sum().sum())
print()
print("Balance de clases (0=benigno, 1=maligno):")
print(df['diagnosis'].value_counts())

In [ ]:
# comparar algunas variables clave entre benigno y maligno
variables = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean']
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for ax, v in zip(axs.ravel(), variables):
    sns.boxplot(data=df, x='diagnosis', y=v, ax=ax)
    ax.set_title(v)
plt.tight_layout()
plt.show()
# Los tumores malignos tienden a tener mayor radio, perimetro y area.

In [ ]:
# correlacion de las variables _mean con el diagnostico
cols_mean = [c for c in df.columns if c.endswith('_mean')] + ['diagnosis']
plt.figure(figsize=(10,8))
sns.heatmap(df[cols_mean].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlacion (variables _mean)')
plt.show()
# radius, perimeter, area, concavity y concave points son las mas discriminativas.

## 2. Preprocesamiento

In [ ]:
X = df.drop(columns='diagnosis')
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# estandarizar
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("train:", X_train.shape, "| test:", X_test.shape)
# Estrategia extra: al haber variables muy correlacionadas (multicolinealidad),
# usamos PCA en el siguiente paso para reducir redundancia.

## 3. Reducción de dimensionalidad (PCA)

In [ ]:
pca = PCA()
pca.fit(X_train_s)
var_acum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,4))
plt.plot(range(1, len(var_acum)+1), var_acum, marker='o')
plt.axhline(0.95, color='r', linestyle='--')
plt.xlabel('componentes'); plt.ylabel('varianza acumulada')
plt.title('PCA - varianza explicada')
plt.show()

n_95 = np.argmax(var_acum >= 0.95) + 1
print("Componentes para 95% de varianza:", n_95)

In [ ]:
# proyeccion 2D para ver separacion de clases
X_pca2 = PCA(n_components=2).fit_transform(X_train_s)
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=y_train, palette='deep')
plt.title('Tumores en 2 componentes principales')
plt.show()
# Las clases se separan bastante bien incluso en solo 2 dimensiones.

## 4. Modelado y evaluación

In [ ]:
# grid de hiperparametros por modelo
configs = {
    'LogisticRegression': (LogisticRegression(max_iter=5000),
                           {'C': [0.1, 1, 10]}),
    'SVM': (SVC(probability=True),
            {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}),
    'RandomForest': (RandomForestClassifier(random_state=42),
                     {'n_estimators': [100, 200], 'max_depth': [None, 5, 10]})
}

resultados = []
mejores = {}
for nombre, (modelo, grid) in configs.items():
    gs = GridSearchCV(modelo, grid, cv=5, scoring='f1')
    gs.fit(X_train_s, y_train)
    y_pred = gs.predict(X_test_s)
    mejores[nombre] = gs.best_estimator_
    resultados.append({
        'modelo': nombre,
        'mejores_params': gs.best_params_,
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred), 4),
        'recall': round(recall_score(y_test, y_pred), 4),
        'f1': round(f1_score(y_test, y_pred), 4)
    })

pd.DataFrame(resultados)

In [ ]:
# matriz de confusion y ROC del mejor modelo (por f1)
mejor_nombre = max(resultados, key=lambda r: r['f1'])['modelo']
mejor_modelo = mejores[mejor_nombre]
y_pred = mejor_modelo.predict(X_test_s)
y_score = mejor_modelo.predict_proba(X_test_s)[:, 1]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=axs[0])
axs[0].set_title(f'Matriz de confusion ({mejor_nombre})')
axs[0].set_xlabel('predicho'); axs[0].set_ylabel('real')

fpr, tpr, _ = roc_curve(y_test, y_score)
axs[1].plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.3f}')
axs[1].plot([0,1],[0,1],'k--')
axs[1].set_xlabel('FPR'); axs[1].set_ylabel('TPR')
axs[1].set_title('Curva ROC')
axs[1].legend()
plt.show()

## 5. Conclusiones y reflexiones

Los tres modelos alcanzan métricas altas (accuracy y F1 sobre 0.95), lo que muestra que el dataset es bastante separable. SVM y Regresión Logística suelen destacar aquí gracias a la estandarización previa.

En un contexto médico el **recall** de la clase maligna es especialmente importante, porque un falso negativo (decir "benigno" cuando es maligno) es el error más grave. Por eso conviene priorizar el modelo con mayor recall/F1 sobre uno con solo buena accuracy.

El preprocesamiento (estandarización) fue clave para SVM y regresión logística, y PCA mostró que la información se concentra en pocas componentes. Como mejoras futuras: validación con más datos, ajuste del umbral de decisión para subir el recall, y probar modelos tipo XGBoost.